# Chapter 2: Multi-Channel Signals & The Power of NumPy ndarrays
*Python for Data Analysis — Applied Engineering Edition*

> **Navigation:** [<< Course Roadmap](./00_roadmap_and_orientation.ipynb) | [<< Previous: Chapter 01](./session_01_environment_ipython.ipynb) | **Current: Chapter 02** | [Next: Chapter 03 >>](./session_03_numpy_filtering.ipynb)

---

## Building Signals: From Single Values to Matrix Grids

In our previous chapter, we saw that Python acts as an expressive "glue" on top of low-level C memory buffers. Vectorization proved to run 30+ times faster than standard Python `for` loops. But *how* does NumPy actually accomplish this behind the scenes?

To work with physical building telemetry—such as reading temperatures across 20 floors or streaming 3-phase electrical power meters—we must master NumPy's core data structure: the **N-dimensional array (`ndarray`)**.

In this chapter, we will unpack how memory is laid out inside an array, how selecting the right numeric `dtype` instantly cuts server RAM consumption in half, and how the **Broadcasting mechanism** lets you calibrate thousands of sensor channels simultaneously without writing a single `for` loop.

### The Big Picture: Calibrating a Sensor Stream in C
Hardware analog-to-digital converters (ADCs) transmit raw integer values between `0` and `4095`. Run the cell below (`Shift + Enter`) to watch how a single broadcasted expression transforms 24 hours of raw multi-channel ADC integers into physical engineering units (°C and kW) in pure C machine code:

In [ ]:
import numpy as np

# 1. Simulate 24 hours of 1-minute readings from 3 building sensors (1440 samples x 3 channels)
# Columns: [Supply Air Temp, Return Air Temp, Fan Motor Power]
np.random.seed(42)
raw_adc = np.random.randint(1800, 3200, size=(1440, 3), dtype=np.int32)

# 2. Physical calibration vectors: [scale, offset] for each channel
# Temperature formula: °C = (adc / 4095) * 50.0 - 5.0
# Power formula:       kW = (adc / 4095) * 15.0
scales = np.array([50.0 / 4095, 50.0 / 4095, 15.0 / 4095])
offsets = np.array([-5.0, -5.0, 0.0])

# 3. Vectorized broadcasted conversion (executed in C without any Python loops)
physical_units = (raw_adc * scales) + offsets

print(f"Raw ADC Matrix shape:        {raw_adc.shape} | dtype: {raw_adc.dtype}")
print(f"Physical Matrix shape:       {physical_units.shape} | dtype: {physical_units.dtype}")
print("\nFirst 3 readings (Supply Air °C, Return Air °C, Fan Power kW):")
print(np.round(physical_units[:3], 2))


---

## 1. The Anatomy of an ndarray: Why It's Blazing Fast

Unlike a standard Python list (which stores pointers to scattered heap objects), a NumPy `ndarray` is a **homogeneous container**: every single number shares the exact same numeric data type.

An `ndarray` consists of two distinct components in memory:
1. **The Raw Data Buffer:** A single, unbroken contiguous block of bytes allocated sequentially in RAM.
2. **The Metadata Descriptor:** A lightweight Python object storing:
   * **Shape (`.shape`):** Dimensions tuple (e.g., `(1440, 3)` = 1440 rows by 3 columns).
   * **Dtype (`.dtype`):** Byte format of each number (e.g., `float64` is 8 bytes, `float32` is 4 bytes).
   * **Strides (`.strides`):** Number of bytes the CPU must step to jump to the next element or next row.

Because data is contiguous, modern CPUs can pull entire rows directly into L1/L2 cache and execute operations in parallel using SIMD hardware instructions.

---

## 2. Choosing the Right Dtype: Cutting RAM Usage in Half

In sensor telemetry, storing millions of observations in standard `float64` (the Python default) is often wasteful:

| Data Type (`dtype`) | Bytes | Value Range | Best Sensor Application |
|---|---|---|---|
| `int16` | 2 bytes | -32,768 to 32,767 | Raw 12-bit / 16-bit ADC hardware inputs |
| `int32` | 4 bytes | -2.14B to 2.14B | High-volume pulse counters, epoch timestamps |
| `float32` | 4 bytes | ~7 decimal digits | Physical metrics (Temp, Humidity, kW) — **halves RAM** |
| `float64` | 8 bytes | ~16 decimal digits | Scientific simulations requiring extreme precision |

You convert between types at any time using the **`.astype()`** method.

Let's see how memory layout and dtypes work in practice:

### A Quick Mental Check: Calculating Array Memory
Examine the code below. We create an array of 100,000 readings across 4 power meters in `float64` (8 bytes per value).

**Question:** How many bytes will this array occupy in RAM? ($100,000 \times 4 \times 8 = ?$). What will happen to memory if we cast it to `float32`?
* Formulate your mental prediction, then run the cell:

In [ ]:
# Mental Prediction: 100,000 * 4 * 8 bytes = 3,200,000 bytes ≈ 3.05 MB
readings_f64 = np.random.uniform(10.0, 100.0, size=(100_000, 4))

print(f"Shape:            {readings_f64.shape}")
print(f"Dtype:            {readings_f64.dtype}")
print(f"Memory (float64): {readings_f64.nbytes / (1024**2):.2f} MB")

# Casting to float32 cuts bytes per element from 8 to 4
readings_f32 = readings_f64.astype(np.float32)
print(f"Memory (float32): {readings_f32.nbytes / (1024**2):.2f} MB (Exactly 50% savings!)")


### Hands-On: Cutting RAM by Choosing int16
Many hardware pulse counters log integer counts. If we know counts never exceed 30,000 per hour, we can store them in `np.int16` (2 bytes) instead of `np.int64` (8 bytes) for a **75% memory reduction**.

In the cell below:
1. Change the dtype in `.astype()` from `np.int32` to `np.int16`.
2. Run the cell and verify that the memory drops to ~0.76 MB.

In [ ]:
pulses = np.random.randint(100, 20_000, size=(100_000, 4), dtype=np.int64)
print(f"Original int64 memory: {pulses.nbytes / (1024**2):.2f} MB")

# Modify np.int32 to np.int16 below:
compact_pulses = pulses.astype(np.int16)
print(f"Optimized int16 memory: {compact_pulses.nbytes / (1024**2):.2f} MB (75% savings!)")


### Hands-On: Managing Multi-Dimensional Sensor Grids
NumPy arrays can have any number of dimensions. Imagine monitoring **5 buildings**, each logging **24 hourly samples** across **4 sensor channels**.

Complete the blanks below (replace `___` with the correct method or dtype):

In [ ]:
# 3D grid: (5 buildings, 24 hours, 4 channels)
grid_f64 = np.random.uniform(18.0, 26.0, size=(5, 24, 4))

# Fill in the blanks:
# 1. Read the total count of numbers using .size
total_readings = grid_f64.size

# 2. Cast the 3D grid to single-precision float32 using .astype(np.float32)
grid_f32 = grid_f64.astype(np.float32)

print(f"Grid shape: {grid_f32.shape}")
print(f"Total readings: {total_readings}")
print(f"Memory in float32: {grid_f32.nbytes} bytes")


### Guided Exercise: Optimizing a Building Portfolio
Now apply your knowledge to calculate and optimize memory for a regional building portfolio.

**Your Task:**
1. Create a 3D array named `portfolio` of shape `(10, 168, 6)` (10 buildings, 168 hours in a week, 6 sub-meters) initialized with random numbers using `np.random.uniform(0.0, 50.0, size=(10, 168, 6))`.
2. Create `portfolio_f32` by casting `portfolio` to `np.float32`.
3. Compute `savings_ratio = portfolio.nbytes / portfolio_f32.nbytes`.
4. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
# [YOUR CODE HERE]
np.random.seed(42)
portfolio = np.random.uniform(0.0, 50.0, size=(10, 168, 6))
portfolio_f32 = portfolio.astype(np.float32)
savings_ratio = portfolio.nbytes / portfolio_f32.nbytes

# --- AUTOMATED TEST VERIFICATION ---
assert portfolio.shape == (10, 168, 6), f"Expected shape (10, 168, 6), got {portfolio.shape}"
assert portfolio_f32.dtype == np.float32, f"Expected dtype float32, got {portfolio_f32.dtype}"
assert savings_ratio == 2.0, f"Expected exact 2.0x savings, got {savings_ratio}"
print(f"[PASS] All checks passed! Portfolio memory reduced from {portfolio.nbytes:,} to {portfolio_f32.nbytes:,} bytes.")


---

## 3. The Magic of Broadcasting Across Sensor Channels

Now suppose you have a matrix of sensor readings with 5 time-steps across 3 different physical sensors: `[Temperature, Humidity, CO2]`.

An instrument technician tells you that each sensor needs a specific calibration offset:
* Sensor 0 (Temp) is drifting by `-0.3°C`.
* Sensor 1 (Humidity) is drifting by `+1.2%`.
* Sensor 2 (CO2) is drifting by `-15.0 ppm`.

How do you apply these three different adjustments across all rows without writing a loop?

This brings us to NumPy's most powerful feature: **Broadcasting**.

### The Trailing Dimensions Rule
When you perform an operation on two arrays with different shapes, NumPy compares their dimensions from **right to left** (the trailing dimensions):
* Array A (Matrix): `(5, 3)`
* Array B (Vector): `   (3,)`

Two dimensions are compatible if:
1. They are **equal**, or
2. One of them is **1**.

Because the trailing dimension `3` matches `3`, NumPy automatically "stretches" Array B across all 5 rows—without duplicating data in RAM!

Run the cell below to see broadcasting calibrate our multi-channel matrix:

In [ ]:
# 5 readings for [Temp °C, Rel Humidity %, CO2 ppm]
sensor_matrix = np.array([
    [21.2, 45.0, 420.0],
    [21.5, 46.2, 435.0],
    [22.1, 44.8, 450.0],
    [21.9, 45.5, 428.0],
    [22.4, 46.0, 460.0]
])

# 1D calibration offset vector (one per channel)
drift_offsets = np.array([-0.3, +1.2, -15.0])  # Shape: (3,)

# Broadcasting handles the math across all rows in parallel:
calibrated_matrix = sensor_matrix + drift_offsets

print("Raw Uncalibrated Matrix:\n", sensor_matrix)
print("\nCalibrated Sensor Matrix:\n", calibrated_matrix)


Now let's see how broadcasting works across channels:

### A Quick Mental Check: Channel Scaling
Examine the 2D array and 1D vector below.
Matrix shape is `(3, 2)` (3 time-steps, 2 channels: `[Voltage, Current]`).
Scale vector is `np.array([10.0, 2.0])` of shape `(2,)`.

**Question:** What will the output shape and values of `matrix * scales` be?
* Formulate your mental prediction, then run the cell:

In [ ]:
matrix = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0]
])
scales = np.array([10.0, 2.0])

# Mental Prediction: column 0 multiplied by 10, column 1 multiplied by 2
# Row 0: [10.0, 4.0], Row 1: [30.0, 8.0], Row 2: [50.0, 12.0]
scaled = matrix * scales
print("Scaled Result:\n", scaled)


### Hands-On: Selective Channel Offsets
Now suppose an engineer determines that **only channel 1 (Humidity)** has a calibration bias of `+5.0%`, while channel 0 (Temp) and channel 2 (CO2) have `0.0` bias.

In the code cell below:
1. Modify `selective_offsets` so that it adds `0.0` to Temp, `5.0` to Humidity, and `0.0` to CO2.
2. Run the cell and verify that only the middle column increases by 5.0:

In [ ]:
# Modify the values below:
selective_offsets = np.array([0.0, 5.0, 0.0])

adjusted = sensor_matrix + selective_offsets
print("Original row 0: ", sensor_matrix[0])
print("Adjusted row 0: ", adjusted[0])


### Hands-On: Broadcasting Along the Time Axis
What if you need to broadcast down rows instead of across columns?
For example, you have 5 time-steps and a time-decay factor for each time-step: `time_decay` has shape `(5,)`.
To broadcast a `(5,)` array against a `(5, 3)` matrix along axis 0, we must reshape `time_decay` into shape `(5, 1)` using `[:, np.newaxis]`.

Complete the blanks below (replace `___` with the correct syntax):

In [ ]:
time_decay = np.array([1.0, 0.9, 0.8, 0.7, 0.6])  # Shape: (5,)

# Complete the reshaping to add an axis so shape becomes (5, 1):
# decay_2d = time_decay[:, np.newaxis]
decay_2d = time_decay[:, np.newaxis]

# Broadcast multiplication down each row:
decayed_sensors = sensor_matrix * decay_2d

print(f"decay_2d shape: {decay_2d.shape}")
print("First row (decay=1.0):\n", decayed_sensors[0])
print("Last row (decay=0.6):\n", decayed_sensors[-1])


### Guided Exercise: Calibrating 4-Channel Telemetry
Now test your broadcasting skills on a 4-channel building telemetry dataset.

You have 100 timesteps of 4 environmental channels: `[Supply Air °C, Return Air °C, Zone Temp °C, Chilled Water Temp °C]`.
Each sensor channel has an individual calibration scale factor: `scales = np.array([1.02, 0.98, 1.01, 0.99])`.

**Your Task:**
1. Multiply `hvac_data` by `scales` using broadcasting to create `calibrated_hvac`.
2. Compute the mean temperature for each channel across all 100 timesteps: `channel_means = calibrated_hvac.mean(axis=0)`.
3. Run the cell to execute the automated verification test (`assert`)!

In [ ]:
np.random.seed(42)
hvac_data = np.random.uniform(15.0, 25.0, size=(100, 4))
scales = np.array([1.02, 0.98, 1.01, 0.99])

# [YOUR CODE HERE]
calibrated_hvac = hvac_data * scales
channel_means = calibrated_hvac.mean(axis=0)

# --- AUTOMATED TEST VERIFICATION ---
assert calibrated_hvac.shape == (100, 4), f"Expected shape (100, 4), got {calibrated_hvac.shape}"
assert channel_means.shape == (4,), f"Expected 4 channel means, got {channel_means.shape}"
# Verify column 0 was scaled by 1.02
assert np.isclose(calibrated_hvac[:, 0], hvac_data[:, 0] * 1.02).all(), "Column 0 scaling incorrect"
print(f"[PASS] All checks passed! Channel means: {np.round(channel_means, 2)}")


---

## 4. Learning Through Deliberate Mistakes

Broadcasting and integer types are powerful, but they have two classic engineering failure modes. Let's trigger them intentionally so you can diagnose them immediately when they occur in production.

### Trap 1: Dimension Mismatch in Broadcasting
What happens if you try to add a 1D array of length 5 (e.g., hourly weights) across a `(5, 3)` matrix along the wrong axis?

NumPy compares trailing dimensions from the right: `3` does not equal `5`! The operation fails with a `ValueError`.

In [ ]:
# Matrix shape: (5, 3)
data = np.zeros((5, 3))
# Time weights shape: (5,)
time_weights = np.array([1.0, 1.1, 1.2, 1.3, 1.4])

# # If you uncomment the line below, it triggers:
# # ValueError: operands could not be broadcast together with shapes (5,3) (5,)
# result = data + time_weights

# THE FIX: We must reshape time_weights from (5,) to (5, 1) so trailing dimensions align:
result = data + time_weights[:, np.newaxis]  # or time_weights.reshape(5, 1)
print("Broadcast resolved cleanly! Output shape:", result.shape)


### Trap 2: Silent Integer Overflow in Sensor Accumulation
Many hardware IoT meters send readings as 16-bit integers (`int16`), which can only store values up to `32,767`.

If you sum high-frequency sensor pulses without upcasting, the number wraps around into negative territory—completely corrupting your energy audit calculations!

In [ ]:
# Two sub-meter pulse readings stored in 16-bit hardware integers:
sensor_pulses = np.array([30_000, 5_000], dtype=np.int16)
print("Naive int16 addition (Overflows into negative!):", sensor_pulses[0] + sensor_pulses[1])

# THE FIX: Always cast to int32 or float64 BEFORE accumulating sensor data:
safe_sum = sensor_pulses.astype(np.int32).sum()
print("Correct int32 accumulated sum:               ", safe_sum)


---

## 5. Real-World Challenge: Calibrating 10 Electrical Sub-Meters

Now let's synthesize everything we've learned in a complete engineering challenge.

You are provided with a 2D telemetry array representing power measurements (in Kilowatts) across **10 electrical sub-meters over 100 timesteps**:

Run the cell below to initialize the dataset:

In [ ]:
import numpy as np
np.random.seed(101)

# Shape: (100 timesteps, 10 meters)
raw_power = np.random.uniform(5.0, 50.0, size=(100, 10))
print(f"Raw power matrix initialized. Shape: {raw_power.shape}")


### Your Engineering Challenge
In the code cell below, complete the following analysis:
1. **Check memory:** Verify `.dtype` and calculate total memory usage of `raw_power`.
2. **Vectorized unit conversion:** Convert the entire matrix from Kilowatts (kW) to Megawatts (MW) in a single vectorized operation: `mw_power = raw_power / 1000.0`.
3. **Compute meter averages:** Calculate the mean power for each of the 10 meters across all 100 timesteps (Hint: compute along `axis=0`): `meter_means = mw_power.mean(axis=0)`.
4. **Correct a faulty transformer:** Meter #4 (column index 3) has a faulty transformer causing its readings to be 15% too high. Construct a 1D correction vector of shape `(10,)` initialized to `1.0`, but with index `3` set to `0.85`.
5. **Apply correction:** Multiply `mw_power` by your correction vector to create `calibrated_mw`.
6. Run the assert test cell below to verify your solution!

In [ ]:
# [YOUR CODE HERE]
# 1. Unit conversion
mw_power = raw_power / 1000.0

# 2. Mean power per meter
meter_means = mw_power.mean(axis=0)

# 3. Correction vector for Meter 4 (index 3)
corrections = np.ones(10)
corrections[3] = 0.85

# 4. Calibrated matrix
calibrated_mw = mw_power * corrections

print("Mean power per meter (MW):", np.round(meter_means, 4))


In [ ]:
# --- AUTOMATED TEST VERIFICATION ---
assert mw_power.shape == (100, 10), "mw_power shape mismatch"
assert np.isclose(mw_power.max(), raw_power.max() / 1000.0), "Unit conversion incorrect"
assert meter_means.shape == (10,), "meter_means should have 10 values"
assert corrections.shape == (10,), "corrections should have 10 values"
assert corrections[3] == 0.85 and corrections[0] == 1.0, "corrections vector incorrectly constructed"
assert np.isclose(calibrated_mw[:, 3], mw_power[:, 3] * 0.85).all(), "Meter 4 calibration failed"
assert np.isclose(calibrated_mw[:, 0], mw_power[:, 0]).all(), "Untouched channels should not change"
print("[SUCCESS] Congratulations! All blank-page engineering assertions passed successfully!")


---

## 6. Key Takeaways & Review Routine

Great work completing Chapter 2! Here are the core insights to retain:

1. An `ndarray` is a contiguous block of C memory paired with lightweight metadata (`shape`, `dtype`, `strides`).
2. Casting from `float64` to `float32` via `.astype(np.float32)` cuts RAM consumption in half with zero meaningful precision loss for physical sensor data.
3. **Broadcasting** matches trailing dimensions from right to left, allowing channel-specific operations without copying data or writing loops.

### Quick Retrieval Check (Task ➔ Code)
Test your active recall before moving forward:
* **Prompt 1:** How do you convert a NumPy array `arr` from `float64` to `float32`?
  * *Answer:* `arr.astype(np.float32)`
* **Prompt 2:** Given a 2D matrix of shape `(1000, 4)`, how do you compute the average reading across rows for each of the 4 columns?
  * *Answer:* `matrix.mean(axis=0)`
* **Prompt 3:** How do you reshape a 1D array `vec` of shape `(100,)` into shape `(100, 1)` so it can broadcast down the rows of a `(100, 5)` matrix?
  * *Answer:* `vec[:, np.newaxis]` (or `vec.reshape(100, 1)`)

### Coming Up in Chapter 3:
In our next session, we will master **Boolean Masking, Views vs. Copies, and Signal Filtering**: how to isolate sensor anomalies, clamp physical spikes with `np.where`, and avoid silent bugs caused by modifying array views.

👉 **Next Step:** Open **[Chapter 03: Boolean Masking, Views vs Copies & Signal Filtering](./session_03_numpy_filtering.ipynb)** to continue.